In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Project 28 — Multilingual Local RAG
## Retrieve in One Language, Answer in Another

**Stack:** Ollama · LangChain · ChromaDB · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb

## Step 1 — Setup Multilingual Corpus

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.schema import Document
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.2)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Documents in multiple languages
docs = [
    Document(page_content="Python is a versatile programming language used for web development, "
        "data science, and artificial intelligence.", metadata={"lang": "en", "id": 1}),
    Document(page_content="Python est un langage de programmation polyvalent utilisé pour le "
        "développement web, la science des données et l'intelligence artificielle.",
        metadata={"lang": "fr", "id": 2}),
    Document(page_content="Python ist eine vielseitige Programmiersprache für Webentwicklung, "
        "Datenwissenschaft und künstliche Intelligenz.",
        metadata={"lang": "de", "id": 3}),
    Document(page_content="Machine learning algorithms learn from data to make predictions. "
        "Common types include supervised, unsupervised, and reinforcement learning.",
        metadata={"lang": "en", "id": 4}),
    Document(page_content="Les algorithmes d'apprentissage automatique apprennent à partir "
        "des données. Types courants: supervisé, non supervisé, et par renforcement.",
        metadata={"lang": "fr", "id": 5}),
]

vectorstore = Chroma.from_documents(docs, embeddings, collection_name="multilingual_rag")
print(f"Indexed {len(docs)} documents in {len(set(d.metadata['lang'] for d in docs))} languages")

## Step 2 — Cross-Lingual Retrieval Test

In [ ]:
# Query in English, retrieve from all languages
query_en = "What is Python used for?"
results = vectorstore.similarity_search_with_score(query_en, k=5)

print(f"Query (EN): {query_en}\n")
for doc, score in results:
    lang = doc.metadata['lang']
    print(f"  [{lang}] score={score:.4f} — {doc.page_content[:60]}...")

# Query in French
query_fr = "Qu'est-ce que l'apprentissage automatique?"
results_fr = vectorstore.similarity_search_with_score(query_fr, k=5)

print(f"\nQuery (FR): {query_fr}\n")
for doc, score in results_fr:
    lang = doc.metadata['lang']
    print(f"  [{lang}] score={score:.4f} — {doc.page_content[:60]}...")

## Step 3 — Answer in Target Language

In [ ]:
translate_answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using the provided context. "
     "IMPORTANT: Always respond in {target_lang}, regardless of the context language."),
    ("human", "Context: {context}\n\nQuestion: {question}\n\nAnswer in {target_lang}:")
])
translate_chain = translate_answer_prompt | llm | StrOutputParser()

# English query → French answer
retrieved = vectorstore.similarity_search(query_en, k=3)
context = "\n".join([d.page_content for d in retrieved])

print("EN question → FR answer:")
answer_fr = translate_chain.invoke({
    "context": context, "question": query_en, "target_lang": "French"
})
print(f"  Q: {query_en}")
print(f"  A: {answer_fr}")

print("\nEN question → DE answer:")
answer_de = translate_chain.invoke({
    "context": context, "question": query_en, "target_lang": "German"
})
print(f"  Q: {query_en}")
print(f"  A: {answer_de}")

## What You Learned
- **Cross-lingual retrieval** with multilingual embeddings
- **Language-controlled generation** — answer in any target language
- **Embedding model behavior** across languages